# Lesson 5 !!! THIS IS A DRAFT VERSION !!!

## Rebuilding GPT from scratch.

This time, we will build a GPT [generative pre-trained transformer](https://en.wikipedia.org/wiki/Generative_pre-trained_transformer) similar to Claude or ChatGPT, but of course, much smaller.

Our program is the following:

1. load and tokenize the dataset,
2. build a batch sampler,
3. train a **bigram** baseline,
4. understand the core idea of **causal self-attention**,
5. Replace the **bigram** to a **decoder-only Transformer**,
6. train it and generate text.

This class will bet different that the previous ones, as we do provide the code. The goal is not only to run the code, but to understand **why each component is there**.

In [ ]:
# if running on google colab, run this cell
%cd /content
!rm -rf mini-course-DL
!git clone https://github.com/AdamArras/mini-course-DL.git
%cd mini-course-DL

# Part I : Preparations

## Imports and configuration

We start with a small set of hyperparameters.  
For notebook work, it is useful to have a quick configuration that runs fast, and a full configuration that we will run at the very end.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

quick_run = True # Flase

if quick_run:
    batch_size = 32
    block_size = 128
    max_iters = 1000
    eval_interval = 200
    learning_rate = 3e-4
    eval_iters = 100
    n_embd = 192
    n_head = 6
    n_layer = 4
    dropout = 0.2
else:
    batch_size = 64
    block_size = 256
    max_iters = 5000
    eval_interval = 500
    learning_rate = 3e-4
    eval_iters = 200
    n_embd = 384
    n_head = 6
    n_layer = 6
    dropout = 0.2

## 1. Read the dataset

In [ ]:
with open("data/shakespeare.txt", "r", encoding="utf-8") as f:
    text = f.read()

print("The dataset has " +str(len(text))+ " characters")
print("Here first 1000 characters:")
print("---------------------------")
print(text[:1000])

## 2. Build the vocabulary

Because this is a **character-level** language model, the vocabulary is simply the sorted list of distinct characters appearing in the text.

As in the previous lessons, we need the follwing functions

- `stoi`: string-to-index
- `itos`: index-to-string
- `encode`: text -> list of integers
- `decode`: list of integers -> text

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

print("vocab size:", vocab_size)
print(chars)

## 3. Encode the whole corpus and split train / validation

- the first 90% for training,
- the last 10% for validation.

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

print("total tokens:", len(data))
print("train tokens:", len(train_data))
print("val tokens:", len(val_data))

## 4. What is the training task?

For language modeling, we want to predict the next token. If the current chunk is:
`x = [t0, t1, t2, t3]`
then the targets are:
`y = [t1, t2, t3, t4]`
So every position tries to predict the next character.

In [ ]:
example_block_size = 8
x = train_data[:example_block_size]
y = train_data[1:example_block_size + 1]

print("x:", x.tolist())
print("y:", y.tolist())
print()

for t in range(example_block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context.tolist()} the target is {target.item()}")

## 5. Batch sampling

Instead of always training on the same prefix, we sample many random chunks from the corpus. Each batch has shape:

- `x`: `(B, T)`
- `y`: `(B, T)`

where:

- `B = batch_size`
- `T = block_size`

In [ ]:
def get_batch(split):
    data_source = train_data if split == "train" else val_data
    ix = torch.randint(len(data_source) - block_size, (batch_size,))
    x = torch.stack([data_source[i:i + block_size] for i in ix])
    y = torch.stack([data_source[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

xb, yb = get_batch("train")
print("xb shape:", xb.shape)
print("yb shape:", yb.shape)
print()
print("first training example x:")
print(xb[0])
print()
print("first training example decoded:")
print("---------------------------")
print(decode(xb[0].tolist())) # print(repr(decode(xb[0].tolist())))

## 6. Loss estimation helper

During training we periodically average the loss over several batches for both train and validation splits.

In [ ]:
@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

# Part II : The bigram baseline

We first consider the bigram model, which predicts the next character using only the current character. This model is not very powerful, but it is the first step toward a true transformer architecture.

## 7. Bigram language model

The entire model is just an embedding table of shape `(vocab_size, vocab_size)`.

For each current token, it directly stores logits for the next token.

In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)  # (B, T, C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits_flat = logits.view(B * T, C)
            targets_flat = targets.view(B * T)
            loss = F.cross_entropy(logits_flat, targets_flat)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, _ = self(idx)
            logits = logits[:, -1, :]           # (B, C)
            probs = F.softmax(logits, dim=-1)   # (B, C)
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            idx = torch.cat((idx, idx_next), dim=1)             # (B, T+1)
        return idx

## 8. Sanity check the bigram model

Before training, the loss should be close to `log(vocab_size)`, because predictions are almost random.

In [ ]:
bigram_model = BigramLanguageModel(vocab_size).to(device)
logits, loss = bigram_model(xb, yb)

print("log(vocab_size) = ", np.log(vocab_size))
print("logits shape:", logits.shape)
print("initial loss:", loss.item())

## 9. Train the bigram model

In [ ]:
optimizer = torch.optim.AdamW(bigram_model.parameters(), lr=1e-2)

for step in range(1000):
    if step % 200 == 0:
        losses = estimate_loss(bigram_model)
        print(f"step {step}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch("train")
    _, loss = bigram_model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

## 10. Generate text with the bigram model

In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
sample = bigram_model.generate(context, max_new_tokens=400)[0].tolist()
print(decode(sample))

The text is usually noisy and short-sighted, which is exactly what we should expect from a model with no real context.

# Part III : Understanding self-attention

The bigram model has no memory beyond the current token.

A Transformer fixes this by letting each position build a representation from the **previous positions in the sequence**.

We now isolate the main idea behind attention.

## 11. A toy example: averaging the past with a lower-triangular mask

Suppose each time step wants access to the average of all previous steps, including itself.

A causal mask is lower triangular because position `t` can only look at positions `<= t`.

In [ ]:
torch.manual_seed(42)

B, T, C = 4, 8, 2
x = torch.randn(B, T, C)

wei = torch.tril(torch.ones(T, T)) # wei is short for weight
wei = wei / wei.sum(1, keepdim=True)
out = wei @ x

print("x shape:", x.shape)
print("wei shape:", wei.shape)
print("out shape:", out.shape)
print()
print("first example, first channel:")
print("input :", x[0, :, 0])
print("output:", out[0, :, 0])

This is already a form of communication across time, but it is very limited:

- every previous token gets a fixed weight,
- the weights do not depend on the content of the sequence.

Attention improves this by making the weights **data-dependent**.

## 12. From fixed averaging to learned attention weights

The key idea is:

- build a **query** vector from each token,
- build a **key** vector from each token,
- compare queries and keys with a dot product,
- normalize with softmax,
- use the resulting weights to mix the **value** vectors.

In [ ]:
torch.manual_seed(42)

B, T, C = 4, 8, 32
head_size = 16

x = torch.randn(B, T, C)
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)      # (B, T, hs)
q = query(x)    # (B, T, hs)
v = value(x)    # (B, T, hs)

wei = q @ k.transpose(-2, -1) * head_size**-0.5   # (B, T, T)
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float("-inf"))
wei = F.softmax(wei, dim=-1)

out = wei @ v

print("k shape:", k.shape)
print("q shape:", q.shape)
print("attention weights shape:", wei.shape)
print("output shape:", out.shape)

# Part IV — Building the Transformer

We now implement the same GPT architecture as in the final script:

- token embeddings,
- positional embeddings,
- causal self-attention,
- feedforward layers,
- residual connections,
- layer normalization,
- final language-model head.

## 14. One attention head

In [ ]:
class Head(nn.Module):
    """One head of causal self-attention."""

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape

        k = self.key(x)                         # (B, T, hs)
        q = self.query(x)                       # (B, T, hs)

        wei = q @ k.transpose(-2, -1) * k.shape[-1] ** -0.5   # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        v = self.value(x)                       # (B, T, hs)
        out = wei @ v                           # (B, T, hs)
        return out

## 15. Multi-head attention

Instead of using a single attention map, we use several heads in parallel.  
Each head can specialize in a different type of relation.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multiple heads of self-attention in parallel."""

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

## 16. Feedforward network

After tokens communicate through attention, each position is processed independently through a small MLP.

In [ ]:
class FeedForward(nn.Module):
    """A position-wise MLP."""

    def __init__(self, n_embd):
        super().__init__()
        dim_ff = 4* n_embd # common choice of the dim of the feed foward layer, as in the original transformer paper
        self.net = nn.Sequential(
            nn.Linear(n_embd, dim_ff),
            nn.ReLU(),
            nn.Linear(dim_ff, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

## 17. Transformer block

A block alternates:

1. **communication** via self-attention,
2. **computation** via the feedforward network.

We also use:

- **residual connections**,
- **pre-layer normalization**.

In [ ]:
class Block(nn.Module):
    """Transformer block: communication followed by computation."""

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

## 18. Full GPT language model

This is a **decoder-only Transformer**:

- token embeddings identify the characters,
- positional embeddings encode the time index,
- a stack of Transformer blocks mixes information causally,
- a final linear layer produces logits over the vocabulary.

In [ ]:
class GPTLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        self.blocks = nn.Sequential(
            *[Block(n_embd, n_head=n_head) for _ in range(n_layer)]
        )
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)                             # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))  # (T, C)
        x = tok_emb + pos_emb                                                 # (B, T, C)

        x = self.blocks(x)                                                    # (B, T, C)
        x = self.ln_f(x)                                                      # (B, T, C)
        logits = self.lm_head(x)                                              # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

## 19. Instantiate the model

In [ ]:
model = GPTLanguageModel().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params/1e6:.3f} M parameters")

## 20. Check one forward pass

In [ ]:
xb, yb = get_batch("train")
logits, loss = model(xb, yb)

print("logits shape:", logits.shape)
print("loss:", loss.item())

## 21. Train the Transformer

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for step in range(max_iters):
    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_loss(model)
        print(f"step {step}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch("train")
    _, loss = model(xb, yb)

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

## 22. Generate from the trained GPT

In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
sample = model.generate(context, max_new_tokens=600)[0].tolist()
print(decode(sample))

# Part IV — Reading the architecture as ideas

## 23. What each piece is doing

### Token embedding
Turns a token id into a trainable vector representation.

### Positional embedding
Adds the notion of order.  
Without it, attention would only see a set of vectors.

### Causal mask
Prevents information leakage from the future.

### Multi-head attention
Lets the model compute several relation patterns in parallel.

### Feedforward network
Transforms each token representation after communication has happened.

### Residual connections
Help optimization and stabilize deep networks.

### LayerNorm
Keeps activations well-behaved across features for each token.

# Part V — Suggested exercises

## 24. Exercises

1. **Shape tracking.**  
   For one batch, print the shapes at every stage inside `GPTLanguageModel.forward`.

2. **Parallel heads.**  
   Replace the Python loop in `MultiHeadAttention` by a more vectorized implementation.

3. **Longer context.**  
   Increase `block_size` and observe the effect on memory and quality.

4. **Capacity scaling.**  
   Compare a small model and a larger model.  
   Track parameter count, train loss, val loss, and sample quality.

5. **Different dataset.**  
   Train the same model on your own corpus.

6. **Pretraining / finetuning idea.**  
   Pretrain on a larger text corpus, then finetune on tiny Shakespeare.

## 25. Final remarks

This notebook rebuilds the same core GPT code as the script version, but in a pedagogical order:

- baseline first,
- attention intuition second,
- full Transformer last.

That ordering makes it easier to understand what problem each new component is solving.